In [77]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:10pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<font color ='red' size='6'><b>ch14. 웹 데이터 수집</b></font>

# 1절. selenium을 이용한동적 웹크롤링 문법
- https://selenium-python.readthedocs.io/ 
- `pip install selenium` `conda install selenium` 中 하나 선택 (아나콘다 프롬프트)
    - 경고 무시 => pip install --upgrade requests ( requests를 최신버전으로 upgrade)
                  pip install urllib3==1.26.18 // conda install urllib3==1.26.18  
                  (pip는 무조건 install conda는 다른 lib version )
    selenium Version: 4.47.0 / requests Version: 2.28.1 / urllib3 Version: 2.7.0

In [4]:
# 동적 웹크롤링 기본 라이브러리
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time                    

In [7]:
dv = webdriver.Chrome() # 
dv.get('http://python.org')

In [30]:
dv = webdriver.Chrome() # 
dv.get('http://python.org')
elem = dv.find_element(By.NAME, 'q')
#By.CLASS_NAME,By.ID,By.CSS_SELECTOR,By.TAG_NAME
#a태그에서 By.LINK_TEXT, By.PARTIAL_LINK_TEXT
elem.clear()
elem.send_keys('pycon')
elem.send_keys(Keys.RETURN)  #엔터를 리턴

In [27]:
dv = webdriver.Chrome() # 
dv.get('http://python.org')
elem = dv.find_element(By.NAME, 'q')
elem.send_keys(Keys.CONTROL,'a') # ctrl +a  
elem.send_keys('pycon')
btn_elem=dv.find_element(By.CSS_SELECTOR,'button#submit') # Go 버튼
btn_elem.click()

In [28]:
dv.close()

In [40]:
result_list=dv.find_elements(By.CSS_SELECTOR,'li>h3>a')
for result in result_list[:5]:
    print('{} - {}'.format(result.text, result.get_attribute('href')))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/
PyCon AU 2019 - https://www.python.org/events/python-events/776/
PyCon NL 2025 - https://www.python.org/events/python-events/2084/


In [48]:
from bs4 import BeautifulSoup  #브라우저가 펼쳐져있어야함
soup = BeautifulSoup(dv.page_source,'html.parser')
result_list=soup.select('li>h3>a')
#len(result_list)
for result in result_list[:3]:
    print("{}-{}".format(result.text, result.attrs.get('href')))  # href에 도메인이 미포함됨

PSF PyCon Trademark Usage Policy-/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette)-/events/python-events/378/
PyCon Australia 2013-/events/python-events/57/


In [54]:
from urllib.parse import urlparse
# https://www.python.org/search/?q=pycon&submit=
current_url=dv.current_url
print('현재 url : ', current_url)
result_parse=urlparse(current_url)
print('url parsing 결과 :', result_parse)
domain = f'{result_parse.scheme}://{result_parse.netloc}'
print('현재 domain:', domain)

현재 url :  https://www.python.org/search/?q=pycon&submit=
url parsing 결과 : ParseResult(scheme='https', netloc='www.python.org', path='/search/', params='', query='q=pycon&submit=', fragment='')
현재 domain: https://www.python.org


In [58]:
soup = BeautifulSoup(dv.page_source,'html.parser')
result_list=soup.select('li>h3>a')
#len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text, 
                         domain+result.attrs.get('href'))) 

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/


In [59]:
dv.close() # 브라우저 종료

# 2절. 동적웹크롤링 예제
## 2.1 다음 뉴스 검색

In [32]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time     
news_list=[] # 뉴스제목과 뉴스 link들을 저장할 list
driver=webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5) #다음페이지가 다 뜰때까지 0.5초 대기

query=input('검색할 단어는?')
driver.find_element(By.CLASS_NAME,'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR,'button[type=submit]').click()
time.sleep(2) #페이지 로딩될 시간 동안 대기하기

#뉴스 탭 클릭
#driver.find_elements(By.CSS_SELECTOR,'ul.list_tab>li')[1].click() #좋은방법이 아니다
driver.find_element(By.LINK_TEXT,'뉴스').click()

검색할 단어는?테슬라


In [9]:
div.item-title > strong.tit-g.clamp-g

In [27]:
bodies = driver.find_elements(By.CSS_SELECTOR,'div.item-title > strong.tit-g.clamp-g')
#len(bodies)
for body in bodies:
    a = body.find_element(By.TAG_NAME,'a')
    title=a.text
    link = a.get_attribute('href')
    #print(title,link)
    news_list.append([title,link])

In [24]:
page_nav=driver.find_element(By.CLASS_NAME,'inner_paging')
# page_nav.text
nex_page=page_nav.find_element(By.LINK_TEXT,"4") # a태그의 text가 2인 a 태그
nex_page.click()
time.sleep(2)

In [31]:
import pandas as pd
pd.DataFrame(news_list,columns=['뉴스제목','링크']).sample()

,뉴스제목,링크
17,[날씨] '입추 매직' 극한 폭염 완화…태풍 '찬홈' 경로는?,http://v.daum.net/v/20260811091413556


## 2-2 다음 뉴스 페이징 처리
 - 위의 예제를 이용하여 원하는 페이지 만큼  뉴스 검색 결과를 받아오기
 

In [52]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time     
news_list=[] # 뉴스제목과 뉴스 link들을 저장할 list
driver=webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5) #다음페이지가 다 뜰때까지 0.5초 대기
query='테슬라'
pages=5
driver.find_element(By.CLASS_NAME,'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR,'button[type=submit]').click()
time.sleep(1) #페이지 로딩될 시간 동안 대기하기
driver.find_element(By.LINK_TEXT,'뉴스').click()

# page_nav.text

for page in range(1,pages+1):
    bodies = driver.find_elements(By.CSS_SELECTOR,'div.item-title > strong.tit-g.clamp-g')
    for body in bodies:
        a = body.find_element(By.TAG_NAME,'a')
        title = a.text
        link = a.get_attribute('href')
        news_list.append([title,link])
    page_nav = driver.find_element(By.CLASS_NAME,'inner_paging')
    nex_page = page_nav.find_element(By.LINK_TEXT,str(page+1)) # a태그의 text가 2인 a 태그
    nex_page.click()
    time.sleep(3)
#driver.close()
news_df=pd.DataFrame(news_list,columns=['뉴스제목','링크'])
display(news_df.head())
print(news_df.shape) 

,뉴스제목,링크
0,"운전대, 페달 없는 차 달린다...테슬라 로보택시 승부수",http://v.daum.net/v/20260818081253030
1,"테슬라, 공중에 뜨는 스포츠카 이달 공개 시연",http://v.daum.net/v/20260818003618268
2,"""테슬라 전기차 독주 끝났다""…'안방 사수' 성공한 車 봤더니",http://v.daum.net/v/20260814104009155
3,"‘담금질’ 기아 EV5, 보급형 테슬라Y 비교 우위 앞선 이유 ‘엔지니어링 승리’",http://v.daum.net/v/20260818094707786
4,"기아 EV5, 테슬라 모델 Y 꺾었다…독일 전기 SUV 평가 1위",http://v.daum.net/v/20260818084339932


(50, 2)


## 2-3 맞춤법 검사기
- 네이버 맞춤법 검사기 이용

In [80]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup 
import time    

In [81]:
driver = webdriver.Chrome()

In [82]:
driver.get('http://www.naver.com')
time.sleep(1)
elem = driver.find_element(By.ID,'query')
elem.send_keys(Keys.CONTROL,'a')
elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(2)
textarea = driver.find_element(By.CLASS_NAME,'txt_gray')

#for문 
textarea.clear()
textarea.send_keys('안뇽하시오 방강습니다. 마싱는 점심되세요')
btn=driver.find_element(By.CLASS_NAME,'btn_check')
btn.click()
time.sleep(2)
result = driver.find_element(By.CSS_SELECTOR,'p._result_text.stand_txt').text
print(result)
#driver.close()


안녕하시오 방강습니다. 마시는 점심 되세요


### 맞춤법검사전.txt파일 -> 맞춤법검사후.txt 로 파일 출력

In [86]:
# fp= open('data/ch14_맞춤법검사전.txt','r',encoding='utf-8')
# text=fp.read()
# fp.close()
# text      --> 잘안쓰는방법

In [99]:
with open('data/ch14_맞춤법검사전.txt','r', encoding='utf-8') as fp:
    text =fp.read()
ready_text_list=[] #300자 기준으로 문장단위로 나눠진 text list
while len(text)>=300:
    temp = text[:300]
    last_dot_index=temp.rfind('. ')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)
print([len(read_text) for read_text in ready_text_list])

[298, 240, 273, 205, 130, 265, 248]


In [101]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup 
import time    
driver = webdriver.Chrome()
driver.get('http://www.naver.com')
time.sleep(0.5)
elem = driver.find_element(By.ID,'query')
elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(0.5)
textarea = driver.find_element(By.CLASS_NAME,'txt_gray')
results =''#맞춤법 검사 완료된 text
for idx, ready_text in enumerate (ready_text_list):
    print(f'검사중...{idx+1}/{len(ready_text_list)}')
    textarea.clear()
    textarea.send_keys(ready_text)
    btn=driver.find_element(By.CLASS_NAME,'btn_check')
    btn.click()
    time.sleep(1)
    result = driver.find_element(By.CSS_SELECTOR,'p._result_text.stand_txt').text
    results+=result + " "

    # soup =BeautifulSoup(driver.page_source,'html.parser')
    # result=soup.select_one('p._result_text.stand_txt').text
    
driver.close()
print(results)
with open('data/ch14_맞춤법후.txt','w') as fp:
    fp.write(results)

검사중...1/7
검사중...2/7
검사중...3/7
검사중...4/7
검사중...5/7
검사중...6/7
검사중...7/7
ai로 작성한 뉴스
지난 8월 15일부터 이어진 집중호우로 거제에는 시간당 최대 124.5mm라는 관측 이래 가장 강한 비가 기록되었습니다. 하루 동안 내린 비로는 지난 2002년 태풍 루사 이후 가장 많은 수준입니다. 짧은 시간에 기록적인 물 폭탄이 집중되면서 배수시설이 마비되었고 산지와 하천, 도심 저지대를 동시에 압박하여 거제시 전역에 초비상이 걸렸습니다. 가장 안타까운 인명 피해는 산사태로 인해 발생했습니다. 17일 새벽 4시 40분쯤 거제시 옥포동의 한 아파트 뒷산 비탈면이 무너지며 토사가 건물 1층을 그대로 덮쳤습니다. 이 사고로 집안에 있던 20대 남성 1명이 심정지 상태로 발견되어 병원으로 옮겨졌으나 끝내 숨졌고, 주민 2명이 다쳤습니다. 통영에서도 산사태로 주택이 매몰되어 주민 1명이 구조되는 등 이번 호우로 남해안 지역에서 사상자가 잇따랐습니다. 또한 옥포동 등 거제 내 5곳에서 산사태가 발생해 주변 도로와 상가가 크게 훼손되었고 차량이 파손되는 등 폭우가 휩쓸고 간 흔적이 곳곳에 남았습니다. 도심 전체의 기능도 사실상 마비되었습니다. 고현동, 장평동, 수월동, 일운면 등 거제 곳곳의 하천이 범람하면서 도심 도로가 강물처럼 물에 잠겼고, 물이 허리까지 차올라 차량 수십 대가 고립되거나 빗물에 떠내려가는 목격담이 이어졌습니다. 고현 시외버스터미널과 거제 식물원 등 주요 시설이 물바다가 되었으며, 배수 역량을 넘어선 비로 인해 시내 외 버스와 택시 운행이 일제히 전면 중단되기도 했습니다. 일운면과 고현동, 장승포동 등에서는 수천 세대에 전기가 끊기는 단전 피해까지 겹쳐 주민들이 암흑 속에서 고립되는 큰 불편을 겪었습니다. 아울러 유치원과 초·중·고등학교 등 교육 시설 40여 곳의 건물과 운동장이 침수되거나 누수 피해를 입어 일부 학교는 학사 운영을 조정하거나 휴업을 결정했습니다. 거제에 위치한 한화 오션과 삼성중공업 등 대

# 3절. 연습문제
  https://papago.naver.com/ 을 통해서 "data/ch14_맞춤법후.txt"파일의 내용을 영문으로 번영하여 파일 출력

In [134]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup 
import time   

In [136]:
# "data/ch14_맞춤법후.txt" 파일 쪼개기
with open('data/ch14_맞춤법후.txt','r') as fp:
    text =fp.read()
ready_text_list=[] #1000자기준 나눈 text list
while len(text)>=1000:
    temp = text[:1000]
    last_dot_index=temp.rfind('. ')
    ready_text_list.append(text[:last_dot_index+1])
    text=text[last_dot_index+1:]
ready_text_list.append(text)
#ready_text_list

In [148]:
# "data/ch14_맞춤법후.txt" 파일 번역
driver = webdriver.Chrome()
driver.get('https://papago.naver.com/')
time.sleep(0.5)
btn = driver.find_element(By.CLASS_NAME,'entry-popup-module-scss-module__UZIxta__close')
if btn:
    btn.click()
output_elems=''
for idx, ready_text in enumerate(ready_text_list):
    print(f'번역진행중...{idx+1}/{len(ready_text_list)}')
    input_elem = driver.find_element(By.CLASS_NAME,'text-translator-module-scss-module__CYJRkW__text-editor')
    input_elem.send_keys(Keys.CONTROL,'a')
    input_elem.send_keys(ready_text)
    time.sleep(3)
    output_elem = driver.find_elements(By.CLASS_NAME,'text-editor-module-scss-module__gKzuvW__dynamic-smd')[1].text
    output_elems +=output_elem + " "
driver.close()
with open('data/ch14_자동화영어번역본.txt','w',encoding='utf-8') as fp:
     fp.write(output_elems)
print('파일 생성 완료됨')

번역진행중...1/2
번역진행중...2/2
파일 생성 완료됨
